In [ ]:
#!/usr/bin/env python3
"""
ENTRENAMIENTO OPTIMIZADO - COMPATIBLE CON TODAS LAS VERSIONES DE OPENCV
========================================================================
"""

import cv2
import os
import numpy as np
from tqdm import tqdm
import albumentations as A
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import json

# =====================================================
# CONFIGURACIÓN
# =====================================================

POSE_DIRS = {
    "sentado": "dataset/pos/sentados",
    "rodillas": "dataset/pos/rodillas",
    "pose_t": "dataset/pos/poset"
}

NEG_DIR = "dataset/negativos1"

OUTPUT_DIR = "models_improved"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# HOG
HOG_WIN_SIZE = (64, 128)
HOG_BLOCK_SIZE = (16, 16)
HOG_BLOCK_STRIDE = (8, 8)
HOG_CELL_SIZE = (8, 8)
HOG_NBINS = 9

# SVM
SVM_C = 1.0
SVM_MAX_ITER = 30000
USE_CLASS_WEIGHTS = True

# AUGMENTATION
AUG_POS = 10
AUG_NEG = 2

# HARD NEGATIVE MINING
USE_HARD_NEGATIVES = True
HARD_NEG_ITERATIONS = 1

# =====================================================
# COMPATIBILIDAD CON OPENCV
# =====================================================

# ✅ FIX: Detectar el flag correcto según la versión de OpenCV
try:
    # OpenCV >= 4.5
    SVM_RAW_OUTPUT_FLAG = cv2.ml.SVM_RAW_OUTPUT
except AttributeError:
    try:
        # OpenCV 4.0-4.4
        SVM_RAW_OUTPUT_FLAG = cv2.ml.SVM.RAW_OUTPUT
    except AttributeError:
        # Fallback: usar el valor numérico directo
        SVM_RAW_OUTPUT_FLAG = 1  # El valor real del flag

print(f"✅ OpenCV version: {cv2.__version__}")
print(f"✅ SVM_RAW_OUTPUT flag: {SVM_RAW_OUTPUT_FLAG}")

# =====================================================
# HOG
# =====================================================

hog = cv2.HOGDescriptor(
    HOG_WIN_SIZE, HOG_BLOCK_SIZE, HOG_BLOCK_STRIDE,
    HOG_CELL_SIZE, HOG_NBINS
)

print(f"🔬 Descriptor size: {hog.getDescriptorSize()}")

# =====================================================
# AUGMENTATION
# =====================================================

augment_pos = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=8, border_mode=cv2.BORDER_REFLECT, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.9),
    A.RandomGamma(gamma_limit=(60, 140), p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MedianBlur(blur_limit=5, p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.4),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=0,
                       border_mode=cv2.BORDER_REFLECT, p=0.6),
    A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.4),
    A.Perspective(scale=(0.02, 0.05), p=0.2),
], p=1.0)

augment_neg = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
], p=1.0)

# =====================================================
# EXTRACCIÓN DE FEATURES
# =====================================================

def extract_features(folder, aug_times=0, label_name="clase", use_pos_aug=True):
    """Extrae features HOG con validación robusta"""
    features = []
    skipped = 0
    
    if not os.path.exists(folder):
        print(f"   ⚠️ Carpeta no encontrada: {folder}")
        return np.array([], dtype=np.float32)
    
    images = [f for f in os.listdir(folder) 
              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    augmenter = augment_pos if use_pos_aug else augment_neg
    
    for img_name in tqdm(images, desc=f"  {label_name}"):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            skipped += 1
            continue
        
        img = cv2.resize(img, HOG_WIN_SIZE)
        
        def process(image):
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
            
            if gray.mean() < 5 or gray.mean() > 250:
                return None
            if gray.std() < 5:
                return None
            
            feat = hog.compute(gray)
            return feat.flatten() if feat is not None and len(feat) > 0 else None
        
        feat = process(img)
        if feat is not None:
            features.append(feat)
        else:
            skipped += 1
        
        for _ in range(aug_times):
            try:
                aug_img = augmenter(image=img)["image"]
                feat = process(aug_img)
                if feat is not None:
                    features.append(feat)
                else:
                    skipped += 1
            except Exception as e:
                skipped += 1
    
    print(f"     ✔️ {len(features)} features | ❌ {skipped} descartadas")
    return np.array(features, dtype=np.float32)

# =====================================================
# HARD NEGATIVE MINING OPTIMIZADO Y COMPATIBLE
# =====================================================

def hard_negative_mining_fast(svm, X_neg_pool, max_hard_negs=500):
    """
    ✅ VERSIÓN OPTIMIZADA Y COMPATIBLE
    """
    print("\n  🔍 Hard Negative Mining (optimizado)...")
    
    if len(X_neg_pool) == 0:
        return np.array([], dtype=np.float32)
    
    print(f"     Evaluando {len(X_neg_pool)} negativos...")
    
    # ✅ BATCH PREDICTION con flag compatible
    _, raw_outputs = svm.predict(X_neg_pool, flags=SVM_RAW_OUTPUT_FLAG)
    
    # Decision values
    decision_values = raw_outputs.flatten()
    
    # Negativos "difíciles" = decision value alto
    hard_indices = np.argsort(decision_values)[::-1][:max_hard_negs]
    
    print(f"     ✅ Encontrados {len(hard_indices)} hard negatives")
    print(f"     Decision values: min={decision_values.min():.3f}, "
          f"max={decision_values.max():.3f}, mean={decision_values.mean():.3f}")
    
    return X_neg_pool[hard_indices]

# =====================================================
# ENTRENAMIENTO
# =====================================================

def train_binary_svm_improved(pose_name, pose_dir, neg_dir, output_dir):
    """Entrena un SVM binario optimizado"""
    
    print(f"\n{'='*70}")
    print(f"🤖 ENTRENANDO MODELO: {pose_name.upper()}")
    print(f"{'='*70}")
    
    # PASO 1: POSITIVOS
    print(f"\n📦 PASO 1: Extrayendo POSITIVOS ({pose_name})")
    X_pos = extract_features(pose_dir, aug_times=AUG_POS, 
                            label_name=f"✅ {pose_name}", use_pos_aug=True)
    
    if len(X_pos) == 0:
        print(f"❌ ERROR: No se encontraron imágenes de {pose_name}")
        return None
    
    print(f"   Total positivos: {len(X_pos)}")
    
    # PASO 2: NEGATIVOS
    print(f"\n📦 PASO 2: Extrayendo NEGATIVOS")
    X_neg_parts = []
    
    print(f"  📁 Otras poses:")
    for other_pose, other_dir in POSE_DIRS.items():
        if other_pose != pose_name:
            X_other = extract_features(other_dir, aug_times=3, 
                                      label_name=f"❌ {other_pose}", use_pos_aug=True)
            if len(X_other) > 0:
                X_neg_parts.append(X_other)
    
    print(f"  📁 Fondos:")
    X_fondo = extract_features(neg_dir, aug_times=AUG_NEG, 
                               label_name="❌ fondos", use_pos_aug=False)
    if len(X_fondo) > 0:
        max_fondos = len(X_pos) * 2
        if len(X_fondo) > max_fondos:
            indices = np.random.choice(len(X_fondo), max_fondos, replace=False)
            X_fondo = X_fondo[indices]
        X_neg_parts.append(X_fondo)
    
    X_neg_all = np.vstack(X_neg_parts) if X_neg_parts else np.array([]).reshape(0, X_pos.shape[1])
    
    target_negs = int(len(X_pos) * 1.5)
    if len(X_neg_all) > target_negs:
        indices = np.random.choice(len(X_neg_all), target_negs, replace=False)
        X_neg_initial = X_neg_all[indices]
    else:
        X_neg_initial = X_neg_all
    
    print(f"   Total negativos iniciales: {len(X_neg_initial)}")
    
    # PASO 3: DATASET
    y_pos = np.ones(len(X_pos), dtype=np.int32)
    y_neg = -np.ones(len(X_neg_initial), dtype=np.int32)
    
    X_train = np.vstack([X_pos, X_neg_initial])
    y_train = np.hstack([y_pos, y_neg])
    
    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]
    
    print(f"\n📊 Dataset inicial: {len(X_neg_initial)}:{len(X_pos)} = {len(X_neg_initial)/len(X_pos):.2f}:1")
    
    # PASO 4: ENTRENAR SVM
    print(f"\n⚙️ PASO 4: Entrenando SVM inicial...")
    
    svm = cv2.ml.SVM_create()
    svm.setType(cv2.ml.SVM_C_SVC)
    svm.setKernel(cv2.ml.SVM_LINEAR)
    svm.setC(SVM_C)
    
    if USE_CLASS_WEIGHTS:
        weight_pos = len(X_train) / (2 * len(X_pos))
        weight_neg = len(X_train) / (2 * len(X_neg_initial))
        class_weights = np.array([weight_neg, weight_pos], dtype=np.float32)
        
        print(f"   Class weights: neg={weight_neg:.3f}, pos={weight_pos:.3f}")
        svm.setClassWeights(class_weights)
    
    svm.setTermCriteria((cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS,
                        SVM_MAX_ITER, 1e-6))
    
    svm.train(X_train, cv2.ml.ROW_SAMPLE, y_train)
    print(f"✅ SVM inicial entrenado")
    
    # PASO 5: HARD NEGATIVE MINING
    if USE_HARD_NEGATIVES and len(X_neg_all) > len(X_neg_initial):
        print(f"\n🔍 PASO 5: Hard Negative Mining...")
        
        # Buscar negativos no usados (versión rápida)
        print(f"     Buscando negativos no usados...")
        
        # Crear hash de arrays para comparación rápida
        used_hashes = set()
        for arr in X_neg_initial:
            # Hash rápido basado en los primeros/últimos valores
            h = hash((arr[0], arr[10], arr[-1], arr.sum()))
            used_hashes.add(h)
        
        unused_indices = []
        for i, arr in enumerate(X_neg_all):
            h = hash((arr[0], arr[10], arr[-1], arr.sum()))
            if h not in used_hashes:
                unused_indices.append(i)
        
        if len(unused_indices) > 0:
            X_neg_unused = X_neg_all[unused_indices]
            print(f"     Negativos no usados: {len(X_neg_unused)}")
            
            for iteration in range(HARD_NEG_ITERATIONS):
                print(f"\n  Iteración {iteration + 1}/{HARD_NEG_ITERATIONS}:")
                
                hard_negs = hard_negative_mining_fast(svm, X_neg_unused, max_hard_negs=500)
                
                if len(hard_negs) > 0:
                    X_train = np.vstack([X_train, hard_negs])
                    y_train = np.hstack([y_train, -np.ones(len(hard_negs), dtype=np.int32)])
                    
                    print(f"     Re-entrenando con {len(hard_negs)} hard negatives...")
                    
                    if USE_CLASS_WEIGHTS:
                        num_pos = (y_train == 1).sum()
                        num_neg = (y_train == -1).sum()
                        weight_pos = len(y_train) / (2 * num_pos)
                        weight_neg = len(y_train) / (2 * num_neg)
                        class_weights = np.array([weight_neg, weight_pos], dtype=np.float32)
                        svm.setClassWeights(class_weights)
                    
                    svm.train(X_train, cv2.ml.ROW_SAMPLE, y_train)
                    print(f"     ✅ Re-entrenamiento completado")
    
    # PASO 6: CROSS-VALIDATION
    print(f"\n📊 PASO 6: Validación Cruzada (5-fold)...")
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_fold_train = X_train[train_idx]
        y_fold_train = y_train[train_idx]
        X_fold_val = X_train[val_idx]
        y_fold_val = y_train[val_idx]
        
        svm_fold = cv2.ml.SVM_create()
        svm_fold.setType(cv2.ml.SVM_C_SVC)
        svm_fold.setKernel(cv2.ml.SVM_LINEAR)
        svm_fold.setC(SVM_C)
        
        if USE_CLASS_WEIGHTS:
            num_pos = (y_fold_train == 1).sum()
            num_neg = (y_fold_train == -1).sum()
            weight_pos = len(y_fold_train) / (2 * num_pos)
            weight_neg = len(y_fold_train) / (2 * num_neg)
            class_weights = np.array([weight_neg, weight_pos], dtype=np.float32)
            svm_fold.setClassWeights(class_weights)
        
        svm_fold.setTermCriteria((cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS,
                                  SVM_MAX_ITER, 1e-6))
        svm_fold.train(X_fold_train, cv2.ml.ROW_SAMPLE, y_fold_train)
        
        _, y_fold_pred = svm_fold.predict(X_fold_val)
        accuracy = (y_fold_val == y_fold_pred.flatten()).sum() / len(y_fold_val)
        cv_scores.append(accuracy)
        
        print(f"   Fold {fold}: {accuracy*100:.2f}%")
    
    print(f"   CV Mean: {np.mean(cv_scores)*100:.2f}% ± {np.std(cv_scores)*100:.2f}%")
    
    # PASO 7: EVALUACIÓN
    print(f"\n📊 PASO 7: Evaluación en Test Set...")
    
    X_pos_test = extract_features(pose_dir, aug_times=0, label_name="test-pos", use_pos_aug=True)
    X_neg_test_parts = []
    
    for other_pose, other_dir in POSE_DIRS.items():
        if other_pose != pose_name:
            X_other = extract_features(other_dir, aug_times=0, label_name=f"test-{other_pose}", use_pos_aug=True)
            if len(X_other) > 0:
                X_neg_test_parts.append(X_other[:min(len(X_other), len(X_pos_test))])
    
    X_neg_test = np.vstack(X_neg_test_parts) if X_neg_test_parts else np.array([]).reshape(0, X_pos_test.shape[1])
    
    if len(X_neg_test) > len(X_pos_test):
        indices = np.random.choice(len(X_neg_test), len(X_pos_test), replace=False)
        X_neg_test = X_neg_test[indices]
    
    X_test = np.vstack([X_pos_test, X_neg_test])
    y_test = np.hstack([np.ones(len(X_pos_test), dtype=np.int32),
                        -np.ones(len(X_neg_test), dtype=np.int32)])
    
    _, y_pred = svm.predict(X_test)
    y_pred = y_pred.flatten().astype(int)
    
    print("\n" + "="*70)
    print("REPORTE DE CLASIFICACIÓN")
    print("="*70)
    print(classification_report(y_test, y_pred,
                                target_names=[f"NO-{pose_name}", f"✅ {pose_name}"],
                                digits=3))
    
    cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
    print("\nMatriz de Confusión:")
    print(f"{'':15} Pred NO  Pred SÍ")
    print(f"Real NO    {cm[0,0]:8}  {cm[0,1]:8}")
    print(f"Real SÍ    {cm[1,0]:8}  {cm[1,1]:8}")
    
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n📈 MÉTRICAS:")
    print(f"   Accuracy:  {accuracy*100:.2f}%")
    print(f"   Precision: {precision*100:.2f}%")
    print(f"   Recall:    {recall*100:.2f}%")
    print(f"   F1-Score:  {f1*100:.2f}%")
    
    # PASO 8: GUARDAR
    model_path = os.path.join(output_dir, f"svm_{pose_name}.yml")
    svm.save(model_path)
    print(f"\n💾 Modelo guardado: {model_path}")
    
    return {
        "pose_name": pose_name,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "cv_mean": float(np.mean(cv_scores))
    }

# =====================================================
# MAIN
# =====================================================

def main():
    print("\n" + "="*70)
    print("🎯 ENTRENAMIENTO OPTIMIZADO")
    print("="*70)
    
    results = []
    
    for pose_name, pose_dir in POSE_DIRS.items():
        result = train_binary_svm_improved(pose_name, pose_dir, NEG_DIR, OUTPUT_DIR)
        if result:
            results.append(result)
    
    print("\n" + "="*70)
    print("✅ COMPLETADO")
    print("="*70)
    
    for result in results:
        print(f"\n🤖 {result['pose_name'].upper()}:")
        print(f"   Accuracy:  {result['accuracy']*100:.1f}%")
        print(f"   Precision: {result['precision']*100:.1f}%")
        print(f"   Recall:    {result['recall']*100:.1f}%")
        print(f"   F1:        {result['f1']*100:.1f}%")
        print(f"   CV:        {result['cv_mean']*100:.1f}%")

if __name__ == "__main__":
    main()